# MCM Problem C — Question 2
Notebook: per-week judges + fan-vote aggregation, predictions, metrics, and uncertainty tables.
Outputs: CSVs in `outputs/` and figures in `outputs/figures/`.

In [11]:
# Imports & config
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# output dirs
OUT = Path('outputs')
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Data file defaults (override as needed)
candidates = [
    '2026_MCM_Problem_C_Data.csv',
    'final_features_for_model.csv',
    'Problem_1.md'
]
season_csv = next((p for p in candidates if Path(p).exists()), None)
print('season_csv ->', season_csv)

season_csv -> 2026_MCM_Problem_C_Data.csv


In [12]:
# Load data (season CSV)
if season_csv is None:
    raise FileNotFoundError('No season CSV found in defaults; set `season_csv` to your file path.')
season_df = pd.read_csv(season_csv)
season_df.columns = [c.strip() for c in season_df.columns]
season_df.head()

,celebrity_name,ballroom_partner,celebrity_industry,celebrity_homestate,celebrity_homecountry/region,celebrity_age_during_season,season,results,placement,week1_judge1_score,...,week9_judge3_score,week9_judge4_score,week10_judge1_score,week10_judge2_score,week10_judge3_score,week10_judge4_score,week11_judge1_score,week11_judge2_score,week11_judge3_score,week11_judge4_score
0,John O'Hurley,Charlotte Jorgensen,Actor/Actress,Maine,United States,50,1,2nd Place,2,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kelly Monaco,Alec Mazo,Actor/Actress,Pennsylvania,United States,29,1,1st Place,1,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Evander Holyfield,Edyta Sliwinska,Athlete,Alabama,United States,42,1,Eliminated Week 3,5,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Rachel Hunter,Jonathan Roberts,Model,NaN,New Zealand,35,1,Eliminated Week 4,4,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Joey McIntyre,Ashly DelGrosso,Singer/Rapper,Massachusetts,United States,32,1,3rd Place,3,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Helper functions: name detection, label resolver, column detection, and uncertainty tables

In [ ]:
def pick_name_columns(df):
    # Return dict with keys: 'celebrity','pro','fallback' if found (column names)
    cols = {c.lower():c for c in df.columns}
    def find(cands):
        for s in cands:
            if s in cols:
                return cols[s]
        return None
    celeb_cands = ['celebrity','celebrity_name','star','contestant','name','celeb','celebrityname']
    pro_cands = ['pro','pro_name','partner','dancer','professional','proname']
    fallback_cands = ['couple_label','pair','team','label']
    res = {
        'celebrity': find(celeb_cands),
        'pro': find(pro_cands),
        'fallback': find(fallback_cands)
    }
    # If fallback not found, try constructing from other uncommon combos
    if res['fallback'] is None:
        for c in df.columns:
            if 'couple' in c.lower() or 'pair' in c.lower() or 'team' in c.lower():
                res['fallback'] = c
                break
    return res

def resolve_couple_label(df, couple_id, name_cols):
    # Ensure couple_id column exists
    if 'couple_id' not in df.columns:
        return f'couple_{couple_id}'
    rows = df[df['couple_id']==couple_id]
    if len(rows)==0:
        return f'couple_{couple_id}'
    row = rows.iloc[0]
    # direct couple_label field
    for c in ['couple_label','label','pair','team']:
        if c in df.columns and pd.notna(row.get(c)) and str(row.get(c)).strip():
            return str(row.get(c)).strip()
    celeb = name_cols.get('celebrity')
    pro = name_cols.get('pro')
    if celeb and pd.notna(row.get(celeb)) and str(row.get(celeb)).strip():
        if pro and pd.notna(row.get(pro)) and str(row.get(pro)).strip():
            return f"{row.get(celeb)} — {row.get(pro)}"
        return str(row.get(celeb)).strip()
    if name_cols.get('fallback') and pd.notna(row.get(name_cols['fallback'])):
        return str(row.get(name_cols['fallback']))
    # last resort
    return f'couple_{couple_id}'

def detect_score_columns(df):
    # Detect judge columns (sum if multiple) and fan vote columns. Return names for J and V or None.
    lc = {c.lower():c for c in df.columns}
    judge_cands = ['judges_score','judge_score','judges','j','score','total_score']
    fan_cands = ['fan_votes','fan_vote','fan','v','votes','vote_share','vote']
    # find explicit single columns first
    J_cols = []
    V_cols = []
    for k,v in lc.items():
        if any(k==cand for cand in judge_cands):
            J_cols.append(v)
        if any(k==cand for cand in fan_cands):
            V_cols.append(v)
    # fallback: find columns containing substrings
    if not J_cols:
        for k,v in lc.items():
            if any(sub in k for sub in ['judge','score']) and 'fan' not in k and 'vote' not in k:
                J_cols.append(v)
    if not V_cols:
        for k,v in lc.items():
            if any(sub in k for sub in ['fan','vote','v']) and 'score' not in k:
                V_cols.append(v)
    return J_cols, V_cols

def vote_share_uncertainty_tables(pF_by_week, season, season_df, eps=1e-8):
    # pF_by_week: dict week_idx -> {week_num, rule_type, active_ids, pF (M x n_active)}
    records = []
    week_summary = []
    couple_agg = defaultdict(list)
    name_cols = pick_name_columns(season_df)
    for wk_idx, pack in sorted(pF_by_week.items(), key=lambda x: int(x[0])):
        week_num = pack.get('week_num', wk_idx)
        rule_type = pack.get('rule_type','')
        active = list(pack.get('active_ids',[]))
        pF = np.array(pack.get('pF'))
        if pF.ndim==1:
            pF = pF.reshape(-1, len(active))
        # compute per-couple statistics
        pF_mean = pF.mean(axis=0)
        pF_q025 = np.quantile(pF, 0.025, axis=0)
        pF_q975 = np.quantile(pF, 0.975, axis=0)
        pF_std = pF.std(axis=0)
        # relative width: coef of variation (std/mean) clipped
        pF_rw = pF_std / (pF_mean + eps)
        for idx, cid in enumerate(active):
            label = resolve_couple_label(season_df, cid, name_cols)
            records.append({
                'season': season, 'week_idx': int(wk_idx),'week_num': int(week_num), 'rule_type': rule_type,
                'couple_id': int(cid),'couple_label': label,
                'pF_mean': float(pF_mean[idx]), 'pF_q025': float(pF_q025[idx]), 'pF_q975': float(pF_q975[idx]),
                'pF_w': float(pF_std[idx]), 'pF_rw': float(pF_rw[idx])
            })
            couple_agg[int(cid)].append(float(pF_rw[idx]))
        week_summary.append({'season':season,'week_idx':int(wk_idx),'week_num':int(week_num),'rule_type':rule_type,'UncWeek': float(np.mean(pF_rw))})
    detail_df = pd.DataFrame.from_records(records)
    week_df = pd.DataFrame.from_records(week_summary)
    couple_df = pd.DataFrame([{'couple_id':cid,'UncCouple':np.mean(vals)} for cid,vals in couple_agg.items()])
    return detail_df, week_df, couple_df

## Weekly aggregation methods and prediction helpers

In [14]:
def safe_numeric_sum(df, cols):
    return df[cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

def prepare_scores(df):
    # detect judge and fan columns
    J_cols, V_cols = detect_score_columns(df)
    if J_cols:
        df['J'] = safe_numeric_sum(df, J_cols)
    else:
        # try columns that look numeric and not votes
        df['J'] = df.select_dtypes(include=[np.number]).iloc[:,0]
    if V_cols:
        df['V'] = safe_numeric_sum(df, V_cols)
    else:
        # If no fan votes, create zeros (will be treated carefully)
        df['V'] = 0.0
    # normalize types to numeric
    df['J'] = pd.to_numeric(df['J'], errors='coerce')
    df['V'] = pd.to_numeric(df['V'], errors='coerce')
    return df

def rank_desc(vals):
    # ranks: 1 = best (largest), larger = worse. Break ties by index (couple_id) for stability
    # vals: pd.Series indexed by couple_id or position
    s = vals.copy()
    # argsort descending -> largest first, assign 1..n
    order = (-s).rank(method='min', na_option='bottom')
    return order.astype(int)

def aggregate_week(df_week, method='rank', judges_save=False):
    # df_week: rows for one week, must contain 'couple_id','J','V'
    active = df_week.dropna(subset=['J','V']).copy()
    active_n = len(active)
    if active_n==0:
        return {'predicted':[],'active_n':0}
    # compute ranks and percents
    rJ = rank_desc(active.set_index('couple_id')['J'])
    rV = rank_desc(active.set_index('couple_id')['V'])
    Jsum = active['J'].sum() if active['J'].sum()>0 else 1.0
    Vsum = active['V'].sum() if active['V'].sum()>0 else 1.0
    pJ = active.set_index('couple_id')['J']/Jsum
    pV = active.set_index('couple_id')['V']/Vsum
    # align indexes
    idx = active.set_index('couple_id').index
    if method=='rank':
        R = rJ + rV
        combined = R
        # higher -> worse; eliminate argmax.
        sorted_by = combined.sort_values(ascending=False)
    elif method=='percent':
        P = pJ + pV
        combined = P
        # lower -> worse; eliminate argmin.
        sorted_by = combined.sort_values(ascending=True)
    else:
        raise ValueError('Unknown method')
    # infer how many eliminated: compare active_n with next week externally; here we return ordered list and let caller pick k
    ordered = list(sorted_by.index)
    # compute bottom-two depending on method
    if method=='rank':
        bottom_two = ordered[:2] if len(ordered)>=2 else ordered
    else:
        bottom_two = ordered[:2] if len(ordered)>=2 else ordered
    # judges save rule: eliminate among bottom_two the one with smaller J
    def apply_save(elim_list):
        if len(elim_list)<=1:
            return elim_list
        a,b = elim_list[0], elim_list[1]
        Ja = active.set_index('couple_id').loc[a,'J']
        Jb = active.set_index('couple_id').loc[b,'J']
        if Ja==Jb:
            # tie: break by V smaller, then by id min
            Va = active.set_index('couple_id').loc[a,'V']
            Vb = active.set_index('couple_id').loc[b,'V']
            if Va==Vb:
                return [min(a,b)]
            return [a if Va>Vb else b]
        return [a if Ja<Jb else b]  # note: smaller J means worse (since rank_desc 1=best). But here we compare raw J: smaller value eliminated

    return {
        'ordered': ordered,
        'combined_series': combined,
        'rJ': rJ.to_dict(), 'rV': rV.to_dict(), 'pJ': pJ.to_dict(), 'pV': pV.to_dict(),
        'bottom_two': bottom_two, 'active_n': active_n, 'apply_save': apply_save
    }

## Run weekly predictions and infer actual left/elim

In [15]:
# Prepare season_df: ensure columns
season_df = prepare_scores(season_df)
# ensure season and week column names
if 'season' not in season_df.columns:
    if 'Season' in season_df.columns:
        season_df.rename(columns={'Season':'season'}, inplace=True)
if 'week' not in season_df.columns and 'week_num' in season_df.columns:
    season_df.rename(columns={'week_num':'week'}, inplace=True)
if 'week' not in season_df.columns and 'week_num' not in season_df.columns:
    raise KeyError('No week column found (expected week or week_num).')
# canonical couple_id column check
if 'couple_id' not in season_df.columns:
    # try to find a numeric id column containing 'couple'
    for c in season_df.columns:
        if 'couple' in c.lower() and season_df[c].dtype in [np.int64, np.int32, np.float64]:
            season_df.rename(columns={c:'couple_id'}, inplace=True)
            break
if 'couple_id' not in season_df.columns:
    raise KeyError('No couple_id column found. Please provide encoded couple_id in CSV.')

season_df['season'] = season_df.get('season',1).astype(int)
season_df['week'] = season_df['week'].astype(int)

# build weekly groups sorted by season, week
grouped = season_df.groupby(['season','week'])
weeks = sorted(list(grouped.groups.keys()))
# build quick mapping of (season,week)->couple_ids present
presence = {k: set(grouped.get_group(k)['couple_id'].dropna().astype(int).unique()) for k in weeks}

rows = []
name_cols = pick_name_columns(season_df)
for i, (s,w) in enumerate(weeks):
    dfw = grouped.get_group((s,w))[['couple_id','J','V']].copy()
    agg_rank = aggregate_week(dfw, method='rank', judges_save=False)
    agg_percent = aggregate_week(dfw, method='percent', judges_save=False)
    # infer k eliminated by comparing presence to next week in same season
    next_key = (s, w+1)
    if next_key in presence:
        k = max(0, len(presence[(s,w)]) - len(presence[next_key]))
    else:
        # final week: no inference
        k = 0
    # choose k from ordered lists
    pred_rank = agg_rank['ordered'][:k] if k>0 else []
    pred_percent = agg_percent['ordered'][:k] if k>0 else []
    # apply judges save for bottom-two scenario: when k==1 and judges_save enabled, use apply_save
    # implement rank+save and percent+save variants
    # rank+save
    if k==1 and len(agg_rank['bottom_two'])==2:
        rank_save = agg_rank['apply_save'](agg_rank['bottom_two'])
    else:
        rank_save = pred_rank
    if k==1 and len(agg_percent['bottom_two'])==2:
        percent_save = agg_percent['apply_save'](agg_percent['bottom_two'])
    else:
        percent_save = pred_percent
    # store one row per predicted elimination slot (if multiple eliminated, store list as JSON)
    rows.append({
        'season': s, 'week_num': w, 'active_n': agg_rank['active_n'],
        'predicted_elim_rank': json.dumps(list(pred_rank)),
        'predicted_elim_percent': json.dumps(list(pred_percent)),
        'predicted_elim_rank_save': json.dumps(list(rank_save)),
        'predicted_elim_percent_save': json.dumps(list(percent_save))
    })

weekly_predictions = pd.DataFrame(rows)
weekly_predictions.to_csv(OUT / 'weekly_predictions.csv', index=False)
weekly_predictions.head()

KeyError: 'No week column found (expected week or week_num).'

## Infer actual left-after-week events and compute metrics

In [ ]:
# Build inferred actual leaves: compare presence in consecutive weeks
left_records = []
for s in sorted(set(season_df['season'].unique())):
    weeks_s = sorted(set(season_df[season_df['season']==s]['week']))
    for i,w in enumerate(weeks_s[:-1]):
        cur = set(season_df[(season_df['season']==s)&(season_df['week']==w)]['couple_id'].astype(int))
        nxt = set(season_df[(season_df['season']==s)&(season_df['week']==weeks_s[i+1])]['couple_id'].astype(int))
        left = cur - nxt
        if len(left)==0:
            left_records.append({'season':s,'week_num':w,'left_after_week':[], 'left_flag':False})
        else:
            left_records.append({'season':s,'week_num':w,'left_after_week':sorted(list(left)), 'left_flag':True})
# merge with weekly_predictions
wp = pd.read_csv(OUT / 'weekly_predictions.csv')
wp2 = wp.merge(pd.DataFrame(left_records), how='left', on=['season','week_num'])
# fill missing left_flag as False (final week etc)
wp2['left_flag'] = wp2['left_flag'].fillna(False)
wp2.to_csv(OUT / 'weekly_predictions_with_actuals.csv', index=False)
wp2.head()

## Metrics: DiffRate, FanElimPct, JudgeElimPct, Rescue rates

In [ ]:
import ast
wp2 = pd.read_csv(OUT / 'weekly_predictions_with_actuals.csv')
# helper to parse predicted lists
def parse_list_cell(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []
wp2['pred_rank'] = wp2['predicted_elim_rank'].map(parse_list_cell)
wp2['pred_percent'] = wp2['predicted_elim_percent'].map(parse_list_cell)
wp2['pred_rank_save'] = wp2['predicted_elim_rank_save'].map(parse_list_cell)
wp2['pred_percent_save'] = wp2['predicted_elim_percent_save'].map(parse_list_cell)
# consider weeks with left_flag==True for DiffRate denominator
comp_pairs = [('pred_rank','pred_percent'),('pred_rank','pred_rank_save'),('pred_percent','pred_percent_save')]
diff_rows = []
for a,b in comp_pairs:
    total = 0
    diff = 0
    for _,r in wp2.iterrows():
        if not r['left_flag']:
            continue
        total += 1
        A = set(parse_list_cell(r[a]))
        B = set(parse_list_cell(r[b]))
        if A != B:
            diff += 1
    diff_rate = diff/total if total>0 else np.nan
    diff_rows.append({'pair':f'{a}_vs_{b}','DiffRate':diff_rate,'n_weeks':total})
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(OUT / 'diff_rate.csv', index=False)
# FanElimPct and JudgeElimPct per predicted elimination
# We'll compute for each method the per-elim stats, flattening weeks where left_flag True
methods = ['pred_rank','pred_percent','pred_rank_save','pred_percent_save']
stat_rows = []
for m in methods:
    vals_fan = []
    vals_j = []
    rescue_fan = []
    rescue_judge = []
    for _,r in wp2.iterrows():
        if not r['left_flag']:
            continue
        s = int(r['season']); w = int(r['week_num'])
        dfw = season_df[(season_df['season']==s)&(season_df['week']==w)][['couple_id','J','V']].dropna(subset=['J','V'])
        if dfw.empty:
            continue
        n = len(dfw)
        # compute ranks for fan and judges
        rV = rank_desc(dfw.set_index('couple_id')['V'])
        rJ = rank_desc(dfw.set_index('couple_id')['J'])
        top_pred = parse_list_cell(r[m])
        if len(top_pred)==0:
            continue
        # if multiple eliminated, handle each but we'll append all values
        for eid in top_pred:
            if int(eid) not in rV.index:
                continue
            fan_pct = (rV.loc[int(eid)] - 1)/(n-1) if n>1 else 0.0
            judge_pct = (rJ.loc[int(eid)] - 1)/(n-1) if n>1 else 0.0
            vals_fan.append(fan_pct)
            vals_j.append(judge_pct)
        # rescue indicators: wJ (lowest J), wV (lowest V)
        wJ = int(dfw.set_index('couple_id')['J'].idxmin())
        wV = int(dfw.set_index('couple_id')['V'].idxmin())
        # If prediction not equal to extreme, count rescue. For multi-elim, we treat rescued if extreme not in predicted set
        pred_set = set(top_pred)
        rescue_fan.append(0 if wV in pred_set else 1)
        rescue_judge.append(0 if wJ in pred_set else 1)
    # aggregate
    if len(vals_fan)>0:
        arr_f = np.array(vals_fan)
        arr_j = np.array(vals_j)
        row = {'method':m,
               'FanElimPct_mean': float(np.nanmean(arr_f)), 'FanElimPct_median': float(np.nanmedian(arr_f)),
               'FanElimPct_IQR': float(np.nanpercentile(arr_f,75)-np.nanpercentile(arr_f,25)),
               'JudgeElimPct_mean': float(np.nanmean(arr_j)), 'JudgeElimPct_median': float(np.nanmedian(arr_j)),
               'JudgeElimPct_IQR': float(np.nanpercentile(arr_j,75)-np.nanpercentile(arr_j,25)),
               'FanRescue_rate': float(np.nanmean(rescue_fan)) if rescue_fan else np.nan,
               'JudgeRescue_rate': float(np.nanmean(rescue_judge)) if rescue_judge else np.nan
              }
    else:
        row = {'method':m,'FanElimPct_mean':np.nan,'FanElimPct_median':np.nan,'FanElimPct_IQR':np.nan,'JudgeElimPct_mean':np.nan,'JudgeElimPct_median':np.nan,'JudgeElimPct_IQR':np.nan,'FanRescue_rate':np.nan,'JudgeRescue_rate':np.nan}
    stat_rows.append(row)
metrics_df = pd.DataFrame(stat_rows)
metrics_df.to_csv(OUT / 'metrics_summary.csv', index=False)
metrics_df

## Visualizations
Scatter pV vs pJ per week, boxplots for FanElimPct/JudgeElimPct, diff rate bar chart.

In [ ]:
# Scatter pV vs pJ for each week in the first season found
first_season = int(season_df['season'].min())
weeks_s = sorted(season_df[season_df['season']==first_season]['week'].unique())
for w in weeks_s:
    dfw = season_df[(season_df['season']==first_season)&(season_df['week']==w)].dropna(subset=['J','V']).copy()
    if dfw.empty: continue
    Jsum = dfw['J'].sum() if dfw['J'].sum()>0 else 1.0
    Vsum = dfw['V'].sum() if dfw['V'].sum()>0 else 1.0
    dfw['pJ'] = dfw['J']/Jsum
    dfw['pV'] = dfw['V']/Vsum
    plt.figure(figsize=(6,6))
    plt.scatter(dfw['pV'], dfw['pJ'], c='gray', alpha=0.6)
    # highlight predicted eliminated by rank and percent for this week if present
    row = wp2[(wp2['season']==first_season)&(wp2['week_num']==w)]
    if not row.empty:
        pr = parse_list_cell(row.iloc[0]['predicted_elim_rank'])
        pp = parse_list_cell(row.iloc[0]['predicted_elim_percent'])
        for cid in pr:
            sub = dfw[dfw['couple_id']==int(cid)]
            if not sub.empty:
                plt.scatter(sub['pV'], sub['pJ'], marker='x', c='red', s=80, label='rank_elim' if cid==pr[0] else '')
        for cid in pp:
            sub = dfw[dfw['couple_id']==int(cid)]
            if not sub.empty:
                plt.scatter(sub['pV'], sub['pJ'], marker='o', facecolors='none', edgecolors='blue', s=100, label='percent_elim' if cid==pp[0] else '')
    plt.xlabel('pV')
    plt.ylabel('pJ')
    plt.title(f'Season {first_season} Week {w} pV vs pJ')
    plt.legend(loc='best')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(FIG / f'scatter_s{first_season}_w{w}.png')
    plt.close()
    # only do first few weeks to limit output
    if w - weeks_s[0] >= 4:
        break

# Boxplots for FanElimPct across methods
metrics_df = pd.read_csv(OUT / 'metrics_summary.csv')
plt.figure(figsize=(6,4))
vals = []
labels = []
for m in ['pred_rank','pred_percent','pred_rank_save','pred_percent_save']:
    row = metrics_df[metrics_df['method']==m]
    if row.empty: continue
    labels.append(m)
    vals.append(row['FanElimPct_mean'].values[0])
plt.bar(labels, vals)
plt.title('FanElimPct mean by method')
plt.savefig(FIG / 'fan_elim_pct_bar.png')
plt.close()
# DiffRate bar chart
diff_df = pd.read_csv(OUT / 'diff_rate.csv')
plt.figure(figsize=(6,4))
plt.bar(diff_df['pair'], diff_df['DiffRate'])
plt.ylabel('DiffRate')
plt.title('DiffRate between method pairs')
plt.savefig(FIG / 'diff_rate.png')
plt.close()
print('Saved example figures to', FIG)

## Uncertainty tables (if `pF_by_week` provided)

In [ ]:
# Try to load pF_by_week from pickle or json if present
pf_path = Path('pF_by_week.pkl')
if not pf_path.exists():
    pf_path = Path('pF_by_week.json')
pF_by_week = None
if pf_path.exists():
    if pf_path.suffix=='.pkl':
        with open(pf_path,'rb') as f:
            pF_by_week = pickle.load(f)
    else:
        with open(pf_path,'r') as f:
            pF_by_week = json.load(f)

if pF_by_week is not None:
    detail_df, week_df, couple_df = vote_share_uncertainty_tables(pF_by_week, season=first_season, season_df=season_df)
    detail_df.to_csv(OUT / 'uncertainty_detail.csv', index=False)
    week_df.to_csv(OUT / 'uncertainty_week.csv', index=False)
    couple_df.to_csv(OUT / 'uncertainty_couple.csv', index=False)
    # plots
    plt.figure(figsize=(8,4))
    for rt, g in week_df.groupby('rule_type'):
        plt.plot(g['week_num'], g['UncWeek'], marker='o', label=str(rt))
    plt.xlabel('week_num')
    plt.ylabel('UncWeek')
    plt.legend()
    plt.title('Uncertainty by week')
    plt.tight_layout()
    plt.savefig(FIG / 'uncertainty_week.png')
    plt.close()
    # top-K uncertain couples
    topk = couple_df.sort_values('UncCouple', ascending=False).head(20)
    plt.figure(figsize=(8,6))
    plt.barh(topk['couple_id'].astype(str), topk['UncCouple'])
    plt.xlabel('UncCouple')
    plt.title('Top uncertain couples')
    plt.tight_layout()
    plt.savefig(FIG / 'uncertainty_topcouples.png')
    plt.close()
else:
    print('No pF_by_week file found; skipping uncertainty tables.')

## Save outputs summary and short conclusions

In [ ]:
print('Wrote CSVs to', OUT)
print('Wrote figures to', FIG)
print('Notebook run complete. Review outputs/ for CSVs and figures.')